# Ejercicio 5 — Programación Orientada a Objetos
## Catálogo de Sismos USGS — Examen Final (Variante A)

**Nombre:** Maria de los Angeles Rojas Granizo  
**Carnet:** 202500859

---

Este notebook usa las clases que implementaste en `poo_sismos.py`.

**Requisito previo:** haber completado todos los métodos de `poo_sismos.py`.

**Entrega:**
- `poo_sismos.py` con tu implementación
- Este notebook ejecutado con todos los outputs visibles

> **Ruta esperada en tu repositorio:**  
> `examenes/python/examen_final/poo_sismos.py`  
> `examenes/python/examen_final/examen_final_poo_a.ipynb`

In [12]:
import pandas as pd

URL_SISMOS = (
    "https://earthquake.usgs.gov/fdsnws/event/1/query"
    "?format=csv"
    "&starttime=2024-01-01"
    "&endtime=2024-06-30"
    "&minmagnitude=5.5"
)

sismos = pd.read_csv(URL_SISMOS)
print(f"Dataset cargado: {len(sismos)} sismos")
sismos.head(3)

Dataset cargado: 183 sismos


,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,...,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource
0,2024-06-29T16:38:46.516Z,-54.0633,7.2302,10.0,5.6,mww,209.0,16.0,18.209,0.88,...,2024-09-04T21:57:44.040Z,Bouvet Island region,earthquake,8.89,1.193,0.086,13.0,reviewed,us,us
1,2024-06-29T07:05:32.855Z,-16.0485,-74.5560,18.0,6.1,mww,138.0,89.0,4.606,0.71,...,2024-09-04T21:57:44.040Z,"34 km SW of Atiquipa, Peru",earthquake,7.96,1.856,0.059,28.0,reviewed,us,us
2,2024-06-28T13:00:39.570Z,-20.6490,-175.6315,10.0,5.7,mww,75.0,38.0,0.654,0.65,...,2024-10-09T03:15:27.209Z,"67 km NNW of Houma, Tonga",earthquake,6.56,1.775,0.069,20.0,reviewed,us,us


In [13]:
# Reutiliza la función clasificar_sismo() del Ejercicio 3 para agregar la columna 'categoria'
def clasificar_sismo(mag):
    if mag < 6.0:
        return 'Moderado-Fuerte'
    elif mag < 7.0:
        return 'Fuerte'
    elif mag < 8.0:
        return 'Mayor'
    else:
        return 'Gran terremoto'

sismos['categoria'] = sismos['mag'].apply(clasificar_sismo)
print("Columna 'categoria' agregada.")
print(sismos['categoria'].value_counts())

Columna 'categoria' agregada.
categoria
Moderado-Fuerte    132
Fuerte              47
Mayor                4
Name: count, dtype: int64


In [14]:
from poo_sismos import Sismo, CatalogoSismos
print("Clases importadas correctamente.")

Clases importadas correctamente.


---
## Parte 1 — Probando la clase `Sismo` (objetos individuales)

El siguiente bloque crea tres objetos `Sismo` representativos y llama a todos sus métodos.  
Verifica que el output tiene sentido con los datos.

In [15]:
# Tres sismos representativos del DataFrame
fila_primero  = sismos.iloc[0]
fila_mayor    = sismos.loc[sismos['mag'].idxmax()]
fila_profundo = sismos.loc[sismos['depth'].idxmax()]

def crear_sismo(fila):
    return Sismo(fila['place'], fila['time'], fila['mag'], fila['depth'], fila['magType'])

s_primero  = crear_sismo(fila_primero)
s_mayor    = crear_sismo(fila_mayor)
s_profundo = crear_sismo(fila_profundo)

for etiqueta, s in [("Primero del dataset", s_primero),
                    ("Mayor magnitud",       s_mayor),
                    ("Mayor profundidad",    s_profundo)]:
    print(f"=== {etiqueta} ===")
    print(f"  {s}")
    print(f"  clasificar()            → {s.clasificar()}")
    print(f"  clasificar_profundidad()→ {s.clasificar_profundidad()}")
    print(f"  es_peligroso()          → {s.es_peligroso()}")
    print()

=== Primero del dataset ===
  Sismo mag=5.60 | Moderado-Fuerte | Superficial | Lugar: Bouvet Island region | Escala: mww
  clasificar()            → Moderado-Fuerte
  clasificar_profundidad()→ Superficial
  es_peligroso()          → False

=== Mayor magnitud ===
  Sismo mag=7.50 | Mayor | Superficial | Lugar: 2024 Noto Peninsula, Japan Earthquake | Escala: mww
  clasificar()            → Mayor
  clasificar_profundidad()→ Superficial
  es_peligroso()          → True

=== Mayor profundidad ===
  Sismo mag=6.50 | Fuerte | Profundo | Lugar: 70 km W of Tarauacá, Brazil | Escala: mww
  clasificar()            → Fuerte
  clasificar_profundidad()→ Profundo
  es_peligroso()          → False



### Tu turno — elige un sismo

Crea un objeto `Sismo` usando **cualquier fila** del DataFrame que tú elijas  
(puede ser el más reciente, uno de Guatemala, el más superficial, etc.).  
Muestra su `descripcion()` y justifica tu elección en un comentario.

In [16]:
# Elige una fila y crea tu propio objeto Sismo
# Hint: sismos.loc[indice] o sismos[sismos['place'].str.contains('Guatemala')]

# TU CÓDIGO AQUÍ
fila_guatemala = sismos[sismos['place'].str.contains('Guatemala', case=False)].iloc[0]

mi_sismo = Sismo(
    lugar=fila_guatemala['place'],
    fecha=fila_guatemala['time'],
    magnitud=fila_guatemala['mag'],
    profundidad=fila_guatemala['depth'],
    tipo_escala=fila_guatemala['magType']
)

print(mi_sismo)
print(f"¿Es peligroso? {mi_sismo.es_peligroso()}")


Sismo mag=6.10 | Fuerte | Intermedio | Lugar: 15 km W of Taxisco, Guatemala | Escala: mww
¿Es peligroso? False


---
## Parte 2 — Construyendo el `CatalogoSismos` con todos los datos

Ahora crea el catálogo con **todos** los sismos del DataFrame (no solo 50) y ejerce los métodos de consulta.

In [17]:
catalogo = CatalogoSismos("Sismos Globales — Semestre 1 de 2024")

for indice, fila in sismos.iterrows():
    catalogo.agregar(Sismo(
        lugar       = fila['place'],
        fecha       = fila['time'],
        magnitud    = fila['mag'],
        profundidad = fila['depth'],
        tipo_escala = fila['magType'],
    ))

print(f"Catálogo construido: {len(catalogo)} sismos")

Catálogo construido: 183 sismos


In [18]:
# ── el_mas_intenso() ────────────────────────────────────────────────────────
mas_intenso = catalogo.el_mas_intenso()
print("Sismo más intenso del catálogo completo:")
print(" ", mas_intenso)
print(f"  ¿Es peligroso? → {mas_intenso.es_peligroso()}")

Sismo más intenso del catálogo completo:
  Sismo mag=7.50 | Mayor | Superficial | Lugar: 2024 Noto Peninsula, Japan Earthquake | Escala: mww
  ¿Es peligroso? → True


In [19]:
# ── filtrar_por_categoria() ──────────────────────────────────────────────────
print("Sismos por categoría (catálogo completo):")
for cat in ['Moderado-Fuerte', 'Fuerte', 'Mayor', 'Gran terremoto']:
    grupo = catalogo.filtrar_por_categoria(cat)
    print(f"  {cat:<18}: {len(grupo)} sismos")
    # Muestra los primeros 2 de cada categoría
    for s in grupo[:2]:
        print(f"      {s}")

Sismos por categoría (catálogo completo):
  Moderado-Fuerte   : 132 sismos
      Sismo mag=5.60 | Moderado-Fuerte | Superficial | Lugar: Bouvet Island region | Escala: mww
      Sismo mag=5.70 | Moderado-Fuerte | Superficial | Lugar: 67 km NNW of Houma, Tonga | Escala: mww
  Fuerte            : 47 sismos
      Sismo mag=6.10 | Fuerte | Superficial | Lugar: 34 km SW of Atiquipa, Peru | Escala: mww
      Sismo mag=6.30 | Fuerte | Intermedio | Lugar: 49 km NNE of Port-Olry, Vanuatu | Escala: mww
  Mayor             : 4 sismos
      Sismo mag=7.20 | Mayor | Superficial | Lugar: 10 km WSW of Atiquipa, Peru | Escala: mww
      Sismo mag=7.40 | Mayor | Superficial | Lugar: 15 km S of Hualien City, Taiwan | Escala: mww
  Gran terremoto    : 0 sismos


In [20]:
# ── resumen() ────────────────────────────────────────────────────────────────
catalogo.resumen()

=== Catálogo: Sismos Globales — Semestre 1 de 2024 ===
Total de sismos: 183
Sismo más intenso: Sismo mag=7.50 | Mayor | Superficial | Lugar: 2024 Noto Peninsula, Japan Earthquake | Escala: mww

Desglose por categorías:
  - Moderado-Fuerte   : 132
  - Fuerte            : 47
  - Mayor             : 4
  - Gran terremoto    : 0


### Tu turno — sismos peligrosos

Usa `filtrar_por_categoria()` y el método `es_peligroso()` para  
**listar todos los sismos que son peligrosos** (Mayor o Gran terremoto, superficiales).  
Muestra cuántos hay y su `descripcion()`.

In [21]:
# Hint: usa filtrar_por_categoria() para obtener 'Mayor' y 'Gran terremoto'
#       luego filtra los que es_peligroso() == True con un for

# TU CÓDIGO AQUÍ
sismos_peligrosos = []
for cat in ['Mayor', 'Gran terremoto']:
    grupo = catalogo.filtrar_por_categoria(cat)
    for s in grupo:
        if s.es_peligroso():
            sismos_peligrosos.append(s)

print(f"Cantidad de sismos peligrosos detectados: {len(sismos_peligrosos)}\n")
for s in sismos_peligrosos:
    print(f"→ {s.descripcion()}")

Cantidad de sismos peligrosos detectados: 4

→ Sismo mag=7.20 | Mayor | Superficial | Lugar: 10 km WSW of Atiquipa, Peru | Escala: mww
→ Sismo mag=7.40 | Mayor | Superficial | Lugar: 15 km S of Hualien City, Taiwan | Escala: mww
→ Sismo mag=7.00 | Mayor | Superficial | Lugar: 128 km WNW of Aykol, China | Escala: mww
→ Sismo mag=7.50 | Mayor | Superficial | Lugar: 2024 Noto Peninsula, Japan Earthquake | Escala: mww


---
## Parte 3 — Verificación automática

Esta celda confirma que tu `clasificar()` produce los mismos resultados  
que la función `clasificar_sismo()` del Ejercicio 3 para **todos** los sismos.

In [22]:
errores = 0
for _, fila in sismos.iterrows():
    s = Sismo("", "", fila['mag'], fila['depth'])
    if s.clasificar() != fila['categoria']:
        errores += 1
        print(f"  ✗ mag={fila['mag']:.2f}  pandas={fila['categoria']}  poo={s.clasificar()}")

if errores == 0:
    print(f"✓ Todos los {len(sismos)} sismos clasifican correctamente.")
else:
    print(f"✗ {errores} sismos con clasificación incorrecta. Revisa tu método clasificar().")

✓ Todos los 183 sismos clasifican correctamente.


---
## Preguntas de interpretación

1. ¿Cuántos sismos peligrosos (`es_peligroso() == True`) encontraste en el catálogo completo?  
   ¿En qué regiones del mundo ocurrieron?

   *Tu respuesta:* Se encontraron 4 sismos peligrosos en total. Ocurrieron en Perú, Taiwán, China y Japón. 

2. ¿Cuántos sismos de categoría `'Gran terremoto'` (mag ≥ 8.0) hay en el semestre?  
   Si no hay ninguno, ¿por qué tiene sentido dada la frecuencia global de ese tipo de evento?

   *Tu respuesta:* Hay 0 sismos en esta categoría. A nivel global, los eventos de magnitud $\ge 8.0$ son extremadamente raros, ocurriendo en promedio solo 1 vez al año en todo el planeta. Dado que este catálogo cubre únicamente un semestre (6 meses) y cuenta con una muestra pequeña ($183$ sismos registrados en total), la probabilidad estadística de capturar un "Gran terremoto" en este periodo de tiempo es sumamente baja.

3. ¿Qué ventaja tiene usar la clase `CatalogoSismos` frente a trabajar directamente  
   con el DataFrame de pandas para este tipo de consultas?

   *Tu respuesta:* Las principales ventajas radican en las buenas prácticas de la Programación Orientada a Objetos (POO) y el análisis de datos:
   - Oculta la complejidad interna de Pandas.
   - Si algo cambia, solo se debe sustituir una línea del código, en lugar de buscar y corregir cada celda o script. 
   - Permite asociar comportamientos específicos a los datos. Un DataFrame trata a los sismos como simples filas de texto y números; la estructura de clases los trata como entidades "Sismo" que "saben" clasificarse a sí mismas, aportando semántica y orden al proyecto.